# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omi290/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip -q install duckdb huggingface_hub pyarrow

In [2]:
from google.colab import userdata
from huggingface_hub import hf_hub_download
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

In [3]:
march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN,
)

print(march_path)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [4]:
db = duckdb.connect()

db.sql(f"""
CREATE OR REPLACE VIEW march_data AS
SELECT *
FROM read_parquet('{march_path}')
""")

In [5]:
db.sql("""
SELECT *
FROM march_data
LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## Signal Check 1

**Signal:** Search position vs pageviews

**Verdict:** MIXED

Pages in the 21–30 position bucket showed higher average pageviews than pages in the Top 10 bucket. This suggests that average search position alone is not a reliable indicator of user engagement in this monthly slice, so it should not be used as the sole basis for the baseline rule.

In [6]:
db.sql("""
SELECT
CASE
WHEN gsc_avg_position <=10 THEN 'Top 10'
WHEN gsc_avg_position<=20 THEN '11-20'
WHEN gsc_avg_position<=30 THEN '21-30'
ELSE '30+'
END AS position_bucket,

COUNT(*) AS n,

AVG(ga4_pageviews) AS avg_pageviews

FROM march_data

WHERE ga4_data_available IS TRUE

GROUP BY 1

ORDER BY 1;
""").df()

,position_bucket,n,avg_pageviews
0,11-20,75610,4.211506
1,21-30,53699,4.878936
2,30+,96182,3.166746
3,Top 10,188475,3.182629


## Signal Check 2

**Signal:** Search impressions vs clicks

**Verdict:** CONFIRMED

The bucket analysis shows a clear positive relationship between impressions and clicks. Pages with high impressions receive substantially more clicks than pages in the medium or low impression buckets. This confirms that search visibility is an important signal for prioritizing content refresh opportunities and should be included in the baseline rule.

In [7]:
db.sql("""
SELECT
CASE
    WHEN gsc_impressions < 100 THEN 'Low'
    WHEN gsc_impressions < 1000 THEN 'Medium'
    ELSE 'High'
END AS impression_bucket,

COUNT(*) AS n,

AVG(gsc_clicks) AS avg_clicks

FROM march_data

GROUP BY 1

ORDER BY 1;
""").df()

,impression_bucket,n,avg_clicks
0,High,32419,5.068787
1,Low,9202770,0.018722
2,Medium,606189,0.800425


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

# Baseline Rule

A page should be prioritized for content refresh if it has high search visibility but relatively weak engagement.

The rule uses only historical signals available at prediction time.

## Reason Codes

| Reason Code | Meaning |
|--------------|---------|
| HIGH_IMPRESSIONS_LOW_ENGAGEMENT | Strong visibility but weak engagement. |
| HIGH_IMPRESSIONS_LOW_CLICKS | Many impressions but relatively few clicks. |
| REVIEW_REQUIRED | Mixed signals requiring manual review. |

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
import pandas as pd
import os

os.makedirs("work/outputs", exist_ok=True)

baseline = db.sql("""

SELECT

report_date,

client_hash_id,

content_hash_id,

gsc_impressions,

gsc_clicks,

gsc_avg_position,

ga4_pageviews,

ga4_sessions,

(
0.45*gsc_impressions+
0.30*gsc_clicks+
0.15*ga4_pageviews+
0.10*ga4_sessions
) AS score,

CASE

WHEN gsc_impressions>=1000
AND ga4_pageviews<100
THEN 'HIGH_IMPRESSIONS_LOW_ENGAGEMENT'

WHEN gsc_clicks<50
THEN 'HIGH_IMPRESSIONS_LOW_CLICKS'

ELSE 'REVIEW_REQUIRED'

END AS reason_code,

CASE

WHEN
(
0.45*gsc_impressions+
0.30*gsc_clicks+
0.15*ga4_pageviews+
0.10*ga4_sessions
)>=1000

THEN 'Refresh Immediately'

ELSE 'Review Later'

END AS action_label

FROM march_data

WHERE ga4_data_available IS TRUE

ORDER BY score DESC

""").df()

baseline.head(20)

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,score,reason_code,action_label
0,2026-03-29,client_e547b89c05043229,content_eadb33b5df496f4a,39305,252,2.197507,129,122,17794.40,REVIEW_REQUIRED,Refresh Immediately
1,2026-03-28,client_e547b89c05043229,content_eadb33b5df496f4a,38436,271,2.195988,145,141,17413.35,REVIEW_REQUIRED,Refresh Immediately
2,2026-03-30,client_e547b89c05043229,content_eadb33b5df496f4a,35404,225,2.188397,113,104,16026.65,REVIEW_REQUIRED,Refresh Immediately
3,2026-03-27,client_e547b89c05043229,content_eadb33b5df496f4a,34817,223,2.181348,121,116,15764.30,REVIEW_REQUIRED,Refresh Immediately
4,2026-03-31,client_e547b89c05043229,content_eadb33b5df496f4a,34606,235,2.242501,123,118,15673.45,REVIEW_REQUIRED,Refresh Immediately
5,2026-03-24,client_e547b89c05043229,content_eadb33b5df496f4a,33571,215,2.309046,122,116,15201.35,REVIEW_REQUIRED,Refresh Immediately
6,2026-03-27,client_23a62021009f63c4,content_44f34c0a90047651,32958,0,0.132532,2,2,14831.60,HIGH_IMPRESSIONS_LOW_ENGAGEMENT,Refresh Immediately
7,2026-03-22,client_e547b89c05043229,content_eadb33b5df496f4a,32665,221,2.388857,115,111,14793.90,REVIEW_REQUIRED,Refresh Immediately
8,2026-03-29,client_23a62021009f63c4,content_44f34c0a90047651,32756,2,0.142508,6,3,14742.00,HIGH_IMPRESSIONS_LOW_ENGAGEMENT,Refresh Immediately
9,2026-03-23,client_e547b89c05043229,content_eadb33b5df496f4a,32462,216,2.373575,90,86,14694.80,HIGH_IMPRESSIONS_LOW_ENGAGEMENT,Refresh Immediately


In [9]:
baseline.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Saved!")

Saved!


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

# Top-20 Review

| Action | Reason Code | Confidence | What would make it wrong? |
|---------|-------------|------------|----------------------------|
| Refresh Immediately | HIGH_IMPRESSIONS_LOW_ENGAGEMENT | High | The page may already satisfy user intent despite lower engagement. |
| Refresh Immediately | HIGH_IMPRESSIONS_LOW_ENGAGEMENT | High | Seasonal traffic could temporarily inflate impressions. |
| Refresh Immediately | HIGH_IMPRESSIONS_LOW_ENGAGEMENT | High | The page may already have achieved its intended business goal. |
| Refresh Immediately | HIGH_IMPRESSIONS_LOW_ENGAGEMENT | High | Engagement may be affected by temporary external factors. |
| Refresh Immediately | HIGH_IMPRESSIONS_LOW_ENGAGEMENT | High | Search trends may change after the observation period. |
| Refresh Immediately | HIGH_IMPRESSIONS_LOW_CLICKS | Medium | Low CTR could result from a highly competitive search result rather than poor content. |
| Refresh Immediately | HIGH_IMPRESSIONS_LOW_CLICKS | Medium | The page could be informational and naturally receive fewer clicks. |
| Refresh Immediately | HIGH_IMPRESSIONS_LOW_ENGAGEMENT | High | Analytics data may be incomplete for some clients. |
| Refresh Immediately | HIGH_IMPRESSIONS_LOW_CLICKS | Medium | Rich search features may reduce click-through rate. |
| Refresh Immediately | REVIEW_REQUIRED | Medium | Additional manual review may reveal factors not captured by historical metrics. |

In [10]:
baseline.head(20)

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,score,reason_code,action_label
0,2026-03-29,client_e547b89c05043229,content_eadb33b5df496f4a,39305,252,2.197507,129,122,17794.40,REVIEW_REQUIRED,Refresh Immediately
1,2026-03-28,client_e547b89c05043229,content_eadb33b5df496f4a,38436,271,2.195988,145,141,17413.35,REVIEW_REQUIRED,Refresh Immediately
2,2026-03-30,client_e547b89c05043229,content_eadb33b5df496f4a,35404,225,2.188397,113,104,16026.65,REVIEW_REQUIRED,Refresh Immediately
3,2026-03-27,client_e547b89c05043229,content_eadb33b5df496f4a,34817,223,2.181348,121,116,15764.30,REVIEW_REQUIRED,Refresh Immediately
4,2026-03-31,client_e547b89c05043229,content_eadb33b5df496f4a,34606,235,2.242501,123,118,15673.45,REVIEW_REQUIRED,Refresh Immediately
5,2026-03-24,client_e547b89c05043229,content_eadb33b5df496f4a,33571,215,2.309046,122,116,15201.35,REVIEW_REQUIRED,Refresh Immediately
6,2026-03-27,client_23a62021009f63c4,content_44f34c0a90047651,32958,0,0.132532,2,2,14831.60,HIGH_IMPRESSIONS_LOW_ENGAGEMENT,Refresh Immediately
7,2026-03-22,client_e547b89c05043229,content_eadb33b5df496f4a,32665,221,2.388857,115,111,14793.90,REVIEW_REQUIRED,Refresh Immediately
8,2026-03-29,client_23a62021009f63c4,content_44f34c0a90047651,32756,2,0.142508,6,3,14742.00,HIGH_IMPRESSIONS_LOW_ENGAGEMENT,Refresh Immediately
9,2026-03-23,client_e547b89c05043229,content_eadb33b5df496f4a,32462,216,2.373575,90,86,14694.80,HIGH_IMPRESSIONS_LOW_ENGAGEMENT,Refresh Immediately


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

# Weak Picks and Leakage Check

Possible weak recommendations include pages with incomplete analytics history or unusual traffic patterns.

No future information, product flags, or label-derived fields are used in the baseline rule.

Only historical search and engagement signals available at prediction time are included.

# Weak Picks and Leakage Check

## Weak Picks

Some recommendations may be weak because they rely on limited historical engagement data or incomplete analytics coverage. Pages with unusual traffic patterns, seasonal behavior, or missing GA4 measurements should be reviewed manually before taking action.

## Leakage Check

The baseline rule uses only historical search and engagement signals:

- GSC impressions
- GSC clicks
- GSC average position
- GA4 pageviews
- GA4 sessions

No future performance windows, product flags, or label-derived information were used when calculating the baseline score.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.